# LLM Classifier Comparison / 大模型分类器对照实验

**Goal / 目标:** Compare four ways of doing the 4-way claim classification, all on the **same retrieved evidence** (loaded from `cache/`, produced by `retrieve_export.ipynb`) for a fair comparison against the DeBERTa classifier.

在**完全相同的检索证据**上(从 `cache/` 加载,由 `retrieve_export.ipynb` 产出)对比四种分类方式,与 DeBERTa 分类器公平对照。

| Method / 方法 | Train? / 训练 | Idea / 思路 |
|---|---|---|
| DeBERTa (reference) | ✅ | encoder 判别式,已在 `final` 里,A=0.5649 |
| Qwen zero-shot | ❌ | 直接问,不给例子 |
| Qwen few-shot (1/2/3) | ❌ | prompt 里给 1–3 个示例(in-context learning) |
| Qwen + LoRA | ✅ | 参数高效微调,用全部训练数据 |

**Design / 设计:** 所有 LLM 变体都走**生成式**(让模型输出标签文本再解析),这样 zero-shot / few-shot / LoRA 三者口径统一、可比。

## 0. Setup / 环境准备

**EN:** Install dependencies, mount Google Drive, pick device. `peft` provides LoRA; `bitsandbytes` enables optional 4-bit loading (QLoRA) if GPU memory is tight.

**中文:** 安装依赖、挂载 Drive、选设备。`peft` 提供 LoRA;`bitsandbytes` 用于可选的 4-bit 量化加载(显存紧张时用 QLoRA)。

In [1]:
!pip install -q transformers peft accelerate bitsandbytes datasets
# torchao 是 Colab 预装的旧版(0.10),会和新版 peft 冲突;LoRA 用不到它,直接卸掉
!pip uninstall -y -q torchao

import os, json, random, time
import numpy as np
import torch

RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Mount Drive (Colab) and locate the cache produced by retrieve_export.ipynb
if "COLAB_GPU" in os.environ:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJ = "/content/drive/MyDrive/climate-claim-verification"
else:
    PROJ = ".."
CACHE_DIR = os.path.join(PROJ, "cache")
print("Cache dir:", CACHE_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00
Device: cuda
Mounted at /content/drive
Cache dir: /content/drive/MyDrive/climate-claim-verification/cache


## 1. Load Retrieved Data / 加载检索结果

**EN:** Load the pre-computed retrieval outputs (claim + selected evidence + gold label) for train and dev. No retrieval is run here — everything comes from the cache, so this notebook starts in seconds.

**中文:** 加载预先算好的检索结果(claim + 选出的证据 + 标签),train / dev 各一份。这里**不跑任何检索**,全部来自 cache,所以秒开。

Each item / 每条格式:`{claim_text, evidence_ids, evidence_texts, label}`

In [2]:
def load_cache(name):
    path = os.path.join(CACHE_DIR, name)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_cache("retrieved_train.json")
dev_data   = load_cache("retrieved_dev.json")
print(f"Train: {len(train_data)}  |  Dev: {len(dev_data)}")

LABELS = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]

# peek one example
_id = next(iter(dev_data))
print("\nSample:", _id)
print(dev_data[_id])

Train: 1228  |  Dev: 154

Sample: claim-752
{'claim_text': '[South Australia] has the most expensive electricity in the world.', 'evidence_ids': ['evidence-67732', 'evidence-572512'], 'evidence_texts': ['[citation needed] South Australia has the highest retail price for electricity in the country.', '"South Australia has the highest power prices in the world".'], 'label': 'SUPPORTS'}


## 2. Shared Utilities / 公共部件

**EN:** Three reusable pieces shared by all methods:
1. `build_evidence_text` — join the selected evidence passages into one context (truncated).
2. `build_prompt` — format claim + evidence (+ optional few-shot examples) into an instruction prompt.
3. `parse_label` — robustly extract one of the four labels from the model's free-text output.
4. `accuracy` — same metric as the DeBERTa classifier, so numbers are directly comparable.

**中文:** 所有方法共用的四个部件:
1. `build_evidence_text` — 把选出的证据拼成一段上下文(截断)。
2. `build_prompt` — 把 claim + 证据(+ 可选的 few-shot 示例)拼成指令 prompt。
3. `parse_label` — 从模型自由生成的文本里稳健地抠出四个标签之一。
4. `accuracy` — 和 DeBERTa 分类器同口径,数字可直接对比。

In [3]:
MAX_EVIDENCE_CHARS = 1200   # keep prompts compact / 控制 prompt 长度

def build_evidence_text(item):
    txts = item.get("evidence_texts", [])
    joined = " ".join(f"[{i+1}] {t}" for i, t in enumerate(txts))
    return joined[:MAX_EVIDENCE_CHARS]

INSTRUCTION = (
    "You are a climate-claim fact checker. Given a claim and retrieved evidence, "
    "classify the claim as exactly one of: SUPPORTS, REFUTES, NOT_ENOUGH_INFO, DISPUTED.\n"
    "- SUPPORTS: evidence supports the claim\n"
    "- REFUTES: evidence contradicts the claim\n"
    "- NOT_ENOUGH_INFO: not enough evidence to decide\n"
    "- DISPUTED: evidence is conflicting / disputed\n"
    "Answer with only the label."
)

def format_one(item):
    return f"Claim: {item['claim_text']}\nEvidence: {build_evidence_text(item)}\nLabel:"

def build_prompt(item, examples=None):
    """examples: list of (item, label) for few-shot; None/[] = zero-shot."""
    parts = [INSTRUCTION, ""]
    for ex_item, ex_label in (examples or []):
        parts.append(format_one(ex_item) + " " + ex_label)
    parts.append(format_one(item))
    return "\n".join(parts)

def parse_label(text):
    up = text.upper()
    # check longer labels first to avoid substring clashes
    for lab in ["NOT_ENOUGH_INFO", "SUPPORTS", "REFUTES", "DISPUTED"]:
        if lab in up:
            return lab
    # fallback: also catch "NOT ENOUGH INFO" with spaces
    if "NOT ENOUGH" in up:
        return "NOT_ENOUGH_INFO"
    return "NOT_ENOUGH_INFO"   # safe default / 兜底

def accuracy(preds, data):
    correct = sum(1 for cid, p in preds.items() if p == data[cid]["label"])
    return correct / len(preds)

def label_distribution(preds):
    from collections import Counter
    return dict(Counter(preds.values()))

## 3. Load Base Model / 加载基座模型

**EN:** Load `Qwen2.5-1.5B-Instruct` (instruct version so it follows the labelling instruction). Small enough for a Colab T4 GPU; used for zero-shot, few-shot, and as the LoRA base.

**中文:** 加载 `Qwen2.5-1.5B-Instruct`(instruct 版才会听从分类指令)。够小,Colab T4 能跑;zero-shot / few-shot / LoRA 都用它当基座。

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("Loaded:", MODEL_NAME)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct


## 4. Inference Core / 推理内核(zero-shot & few-shot 共用)

**EN:** One function `run_llm(shot)` drives both zero-shot and few-shot. It builds a prompt (with `shot` in-context examples sampled from the train set, one per class when possible), generates a short output, parses the label, and returns per-claim predictions + accuracy. `shot=0` is zero-shot; `shot=1/2/3` is few-shot.

**中文:** 一个函数 `run_llm(shot)` 同时驱动 zero-shot 和 few-shot。它构造 prompt(从训练集采 `shot` 个 in-context 示例,尽量每类一个),生成简短输出,解析标签,返回逐条预测 + 准确率。`shot=0` 即 zero-shot;`shot=1/2/3` 即 few-shot。

**Note / 注意:** few-shot 示例覆盖不同类别,避免模型偏向某一类。

In [5]:
# Pre-select few-shot example pool: try to get diverse labels
def pick_examples(shot, exclude_id=None):
    if shot <= 0:
        return []
    pool = [(cid, it) for cid, it in train_data.items() if cid != exclude_id]
    # try one distinct label each, then fill
    chosen, used_labels = [], set()
    for cid, it in pool:
        lab = it["label"]
        if lab not in used_labels:
            chosen.append((it, lab)); used_labels.add(lab)
        if len(chosen) >= shot:
            break
    # if still short, fill with any
    i = 0
    while len(chosen) < shot and i < len(pool):
        it = pool[i][1]; chosen.append((it, it["label"])); i += 1
    return chosen[:shot]

@torch.no_grad()
def generate_label(prompt):
    msgs = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=8, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)

def run_llm(shot=0, data=None, verbose_every=40):
    data = data if data is not None else dev_data
    examples = pick_examples(shot)   # fixed example set for all claims
    preds = {}
    t0 = time.time()
    for i, (cid, item) in enumerate(data.items(), 1):
        prompt = build_prompt(item, examples=examples)
        raw = generate_label(prompt)
        preds[cid] = parse_label(raw)
        if i % verbose_every == 0:
            print(f"  {i}/{len(data)} | {time.time()-t0:.0f}s")
    acc = accuracy(preds, data)
    print(f"shot={shot} | accuracy={acc:.4f} | dist={label_distribution(preds)}")
    return preds, acc

## 5. Zero-shot Baseline / 零样本基线

**EN:** No training, no examples — just the instruction + claim + evidence. This is the LLM lower bound.

**中文:** 不训练、不给例子——只有指令 + claim + 证据。这是 LLM 的下限参照。

In [6]:
results = {}
_, results["zero-shot"] = run_llm(shot=0)

  40/154 | 9s
  80/154 | 16s
  120/154 | 24s
shot=0 | accuracy=0.4740 | dist={'REFUTES': 63, 'SUPPORTS': 57, 'DISPUTED': 3, 'NOT_ENOUGH_INFO': 31}


## 6. Few-shot (1 / 2 / 3-shot) / 少样本

**EN:** Add 1–3 labelled examples into the prompt (in-context learning, still no training). See whether more examples help — expect a small, possibly unstable gain over zero-shot.

**中文:** 在 prompt 里加 1–3 个带标签示例(in-context learning,仍不训练)。观察加例子有没有用——预期比 zero-shot 略好但可能不稳。

In [7]:
for k in [1, 2, 3]:
    _, results[f"{k}-shot"] = run_llm(shot=k)

print("\nSo far:", {m: round(a, 4) for m, a in results.items()})

  40/154 | 11s
  80/154 | 23s
  120/154 | 35s
shot=1 | accuracy=0.4545 | dist={'SUPPORTS': 55, 'DISPUTED': 26, 'NOT_ENOUGH_INFO': 57, 'REFUTES': 16}
  40/154 | 16s
  80/154 | 32s
  120/154 | 48s
shot=2 | accuracy=0.4610 | dist={'SUPPORTS': 73, 'DISPUTED': 12, 'NOT_ENOUGH_INFO': 57, 'REFUTES': 12}
  40/154 | 18s
  80/154 | 35s
  120/154 | 53s
shot=3 | accuracy=0.4416 | dist={'SUPPORTS': 63, 'DISPUTED': 17, 'REFUTES': 32, 'NOT_ENOUGH_INFO': 42}

So far: {'zero-shot': 0.474, '1-shot': 0.4545, '2-shot': 0.461, '3-shot': 0.4416}


## 7. LoRA Fine-tuning (r=4 vs r=8) / LoRA 微调 + rank 消融

**EN:** Parameter-efficient fine-tuning, framed as instruction generation (target = label text). LoRA adapters attach to attention `q_proj`/`v_proj`; only these small matrices train, base stays frozen. We compare **rank r=4 vs r=8** as an ablation — smaller r means fewer trainable params (less overfit risk, cheaper), larger r means more capacity. For a fair comparison `lora_alpha = 2·r` keeps the scaling ratio constant.

**中文:** 参数高效微调,当成指令生成(目标=标签文本)。LoRA 挂在注意力 `q_proj`/`v_proj`,只训小矩阵,基座冻结。这里做 **r=4 vs r=8 的消融对比** —— r 小则可训参数少(不易过拟合、更省),r 大则容量更强。为公平对比,`lora_alpha = 2·r` 保持缩放比一致。

**Why compare r / 为什么比 rank:** 数据量小(1228),小 r 可能已够用;跑两组能得出"这个任务需不需要更大适配容量"的结论。

**关键超参:** `r∈{4,8}`, `lora_alpha=2r`, `epochs=3`, `lr=2e-4`, `target_modules=[q_proj,v_proj]`。

In [8]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import gc

# ---- Build training text: prompt + target label, mask the prompt in loss ----
def build_train_text(item):
    msgs = [{"role": "user", "content": build_prompt(item)}]
    prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full_text = prompt_text + " " + item["label"] + tokenizer.eos_token
    return prompt_text, full_text

class LoRADataset(Dataset):
    def __init__(self, data, max_len=512):
        self.items = list(data.values()); self.max_len = max_len
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        item = self.items[idx]
        prompt_text, full_text = build_train_text(item)
        full = tokenizer(full_text, truncation=True, max_length=self.max_len)
        plen = len(tokenizer(prompt_text, truncation=True, max_length=self.max_len)["input_ids"])
        labels = list(full["input_ids"])
        for i in range(min(plen, len(labels))):   # mask prompt tokens in loss
            labels[i] = -100
        full["labels"] = labels
        return full

def collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id or tokenizer.eos_token_id
    input_ids, attn, labels = [], [], []
    for b in batch:
        n = maxlen - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [pad]*n)
        attn.append(b["attention_mask"] + [0]*n)
        labels.append(b["labels"] + [-100]*n)
    return {"input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attn),
            "labels": torch.tensor(labels)}

@torch.no_grad()
def gen_with(m, prompt):
    msgs = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(m.device)
    out = m.generate(**inputs, max_new_tokens=8, do_sample=False,
                     pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def train_and_eval_lora(r, epochs=3, lr=2e-4, batch=2):
    print(f"\n===== LoRA r={r} (alpha={2*r}) =====")
    # fresh base each run to avoid adapter stacking
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
    cfg = LoraConfig(r=r, lora_alpha=2*r, lora_dropout=0.05,
                     target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
    base.config.use_cache = False
    base.gradient_checkpointing_enable()
    pm = get_peft_model(base, cfg)
    pm.enable_input_require_grads()   # 梯度检查点下必需
    pm.print_trainable_parameters()

    loader = DataLoader(LoRADataset(train_data), batch_size=batch, shuffle=True, collate_fn=collate)
    optim = AdamW([p for p in pm.parameters() if p.requires_grad], lr=lr)

    pm.train()
    for ep in range(1, epochs+1):
        tot, t0 = 0.0, time.time()
        for step, b in enumerate(loader, 1):
            b = {k: v.to(pm.device) for k, v in b.items()}
            loss = pm(**b).loss
            loss.backward(); optim.step(); optim.zero_grad()
            tot += loss.item()
            if step % 80 == 0:
                print(f"  ep{ep} {step}/{len(loader)} loss={tot/step:.4f} {time.time()-t0:.0f}s")
        print(f"  epoch {ep} mean loss={tot/len(loader):.4f}")

    # evaluate on dev
    pm.eval()
    pm.config.use_cache = True
    preds = {}
    for cid, item in dev_data.items():
        preds[cid] = parse_label(gen_with(pm, build_prompt(item)))
    acc = accuracy(preds, dev_data)
    print(f"  r={r} dev accuracy={acc:.4f} | dist={label_distribution(preds)}")

    # cleanup GPU memory before next run
    del pm, base, optim, loader
    gc.collect(); torch.cuda.empty_cache()
    return acc, preds

## 8. Run LoRA r=4 & r=8 / 跑两组 LoRA

**EN:** Train and evaluate both ranks, store results. Same inference core & accuracy metric as the other methods.

**中文:** 训练并评估两个 rank,存结果。推理内核和 accuracy 口径与其他方法一致。

**Memory note / 显存提示:** 每个 r 会重新加载一次基座并在结束后释放显存;若 T4 仍 OOM,把基座改成 4-bit 量化加载(QLoRA)或减小 `batch`。

In [9]:
# free the zero/few-shot base model to make room for LoRA training
import gc
try:
    del model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()
print("Freed zero/few-shot model. Starting LoRA runs...")

for r in [4, 8]:
    acc, _ = train_and_eval_lora(r)
    results[f"LoRA r={r}"] = acc

print("\nResults so far:", {m: round(a, 4) for m, a in results.items()})

Freed zero/few-shot model. Starting LoRA runs...

===== LoRA r=4 (alpha=8) =====


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 544,768 || all params: 1,544,259,072 || trainable%: 0.0353
  ep1 80/614 loss=0.7473 40s
  ep1 160/614 loss=0.5435 78s
  ep1 240/614 loss=0.4820 118s
  ep1 320/614 loss=0.4433 158s
  ep1 400/614 loss=0.4181 196s
  ep1 480/614 loss=0.4072 235s
  ep1 560/614 loss=0.3961 275s
  epoch 1 mean loss=0.3877
  ep2 80/614 loss=0.2923 39s
  ep2 160/614 loss=0.3034 78s
  ep2 240/614 loss=0.3082 118s
  ep2 320/614 loss=0.3025 158s
  ep2 400/614 loss=0.3056 198s
  ep2 480/614 loss=0.3065 237s
  ep2 560/614 loss=0.3067 276s
  epoch 2 mean loss=0.3054
  ep3 80/614 loss=0.2810 38s
  ep3 160/614 loss=0.2766 79s
  ep3 240/614 loss=0.2817 117s
  ep3 320/614 loss=0.2840 156s
  ep3 400/614 loss=0.2816 195s
  ep3 480/614 loss=0.2824 235s
  ep3 560/614 loss=0.2783 275s
  epoch 3 mean loss=0.2807
  r=4 dev accuracy=0.5260 | dist={'SUPPORTS': 107, 'REFUTES': 43, 'NOT_ENOUGH_INFO': 4}

===== LoRA r=8 (alpha=16) =====


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705
  ep1 80/614 loss=0.6593 41s
  ep1 160/614 loss=0.5063 80s
  ep1 240/614 loss=0.4367 119s
  ep1 320/614 loss=0.4158 160s
  ep1 400/614 loss=0.3976 198s
  ep1 480/614 loss=0.3852 236s
  ep1 560/614 loss=0.3767 275s
  epoch 1 mean loss=0.3721
  ep2 80/614 loss=0.3022 40s
  ep2 160/614 loss=0.3163 79s
  ep2 240/614 loss=0.3124 118s
  ep2 320/614 loss=0.3041 158s
  ep2 400/614 loss=0.3019 196s
  ep2 480/614 loss=0.3029 235s
  ep2 560/614 loss=0.3073 274s
  epoch 2 mean loss=0.3041
  ep3 80/614 loss=0.2455 39s
  ep3 160/614 loss=0.2477 80s
  ep3 240/614 loss=0.2594 119s
  ep3 320/614 loss=0.2673 157s
  ep3 400/614 loss=0.2647 197s
  ep3 480/614 loss=0.2692 234s
  ep3 560/614 loss=0.2685 273s
  epoch 3 mean loss=0.2660
  r=8 dev accuracy=0.4610 | dist={'NOT_ENOUGH_INFO': 106, 'SUPPORTS': 46, 'DISPUTED': 2}

Results so far: {'zero-shot': 0.474, '1-shot': 0.4545, '2-shot': 0.461, '3-shot': 0.4416, 'LoRA r=4': 0.526,

## 9. Results Summary / 结果汇总

**EN:** Put every method side by side, including the DeBERTa reference (0.5649 from `final`). The story to tell: does in-context learning help, and does LoRA fine-tuning close the gap to the encoder?

**中文:** 把所有方法并排(含 DeBERTa 参照 0.5649,来自 `final`)。要讲的故事:in-context learning 有没有用?LoRA 微调能否追平 encoder?

**Expected / 预期:** zero-shot 最低 → few-shot 略升 → LoRA 明显提升,大概率接近但未必超过 DeBERTa —— 结论:小数据判别任务上,专用 encoder 仍具性价比优势。

In [10]:
DEBERTA_ACC = 0.5649   # reference from final_retrieve_classify.ipynb

print(f"{'Method':<16}{'Accuracy':>10}")
print("-"*26)
print(f"{'DeBERTa (ref)':<16}{DEBERTA_ACC:>10.4f}")
for m in ["zero-shot", "1-shot", "2-shot", "3-shot", "LoRA r=4", "LoRA r=8"]:
    if m in results:
        print(f"{m:<16}{results[m]:>10.4f}")

# save results
os.makedirs(os.path.join(PROJ, "results"), exist_ok=True)
with open(os.path.join(PROJ, "results", "llm_comparison.json"), "w") as f:
    json.dump({**results, "DeBERTa": DEBERTA_ACC}, f, indent=2)
print("\nSaved results/llm_comparison.json")

Method            Accuracy
--------------------------
DeBERTa (ref)       0.5649
zero-shot           0.4740
1-shot              0.4545
2-shot              0.4610
3-shot              0.4416
LoRA r=4            0.5260
LoRA r=8            0.4610

Saved results/llm_comparison.json
